# SST-2 SAE feature enrichment

In [1]:
import json
from pathlib import Path

import torch
import nltk
from IPython.display import display
from nltk.corpus import stopwords

from murano.dataset import LabeledDataset
from murano.plotting import plot_sae_feature_logit_effects, plot_sae_token_activations

DATASET, SPLIT = "stanfordnlp/sst2", "train"
MODEL_ID = "openai-community/gpt2"
SAE_RELEASE = "gpt2-small-res-jb"
SAE_ID = "blocks.8.hook_resid_pre"
N_EXAMPLES, TOP_FEATURES, K_CONTEXTS = 1024, 20, 10
OUTPUT_DIR = "examples/artifacts/murano_sst2_gpt2_sae_viz"

NLTK_DATA_DIR = Path(OUTPUT_DIR) / "nltk_data"
nltk.data.path.append(str(NLTK_DATA_DIR))
try:
    STOPWORDS = set(stopwords.words("english"))
except LookupError:
    NLTK_DATA_DIR.mkdir(parents=True, exist_ok=True)
    nltk.download("stopwords", download_dir=str(NLTK_DATA_DIR), quiet=True)
    STOPWORDS = set(stopwords.words("english"))

## Ranking features by label-aligned token evidence

We rank features by how well their activations align with label-specific token evidence. Intuitively, a positive feature should activate on positive-evidence tokens in positive examples, while a negative feature should do the symmetric thing for negative examples.

### 1. Readable-token mask

We first keep only readable tokens. This removes BOS tokens, stopwords, very short fragments, punctuation-heavy tokens, and GPT-2 continuation pieces.

This step is useful because punctuation and token fragments can sometimes activate strongly, but they are usually not very informative as explanations.

### 2. Label-token weight

For each readable token (t), we compute a smoothed log-odds score:

$$ w(t)=\log\frac{\operatorname{count}_{pos}(t)+1}{\operatorname{total}_{pos}+V}-\log\frac{\operatorname{count}_{neg}(t)+1}{\operatorname{total}_{neg}+V}, $$

where $\operatorname{total}_{pos}$ and $\operatorname{total}_{neg}$ are the total numbers of readable tokens in positive and negative examples, and $V$ is the number of unique readable tokens.

The $+1$ and $+V$ terms implement add-one smoothing, which prevents rare tokens from receiving overly extreme scores. This gives us a data-driven estimate of whether each token provides positive or negative evidence in this SST-2 slice, without relying on a hand-written sentiment lexicon.

### 3. Sentence-level selectivity

For each feature $f$, we pool its readable-token activations within each sentence, then compare the pooled activations across labels:

$$ \Delta(f)=\operatorname{mean}_{pos}(f)-\operatorname{mean}_{neg}(f), \qquad \operatorname{effect}(f)=\frac{\Delta(f)}{\operatorname{std}(f)} $$

Here, $\Delta(f)$ measures the raw positive-minus-negative activation gap, while $\operatorname{effect}(f)$ standardizes that gap by the overall variability of the feature.

We also compute `pos_rate` and `neg_rate`, which measure how often the feature fires in positive and negative examples. This helps prefer features that are consistently label-selective, rather than features that only spike on a few high-activation tokens.

### 4. Token contribution score

We measure token-level evidence by multiplying the feature activation by the token’s label weight:

$$ \operatorname{contrib}_{pos}(i,f)=a_{i,f}\max(w(t_i),0), \qquad \operatorname{contrib}_{neg}(i,f)=a_{i,f}\max(-w(t_i),0) $$

Positive contributions come from activations on tokens with positive label evidence. Negative contributions are defined symmetrically using tokens with negative label evidence.

The final ranking score combines token contribution, the class activation gap, signed effect size, and a penalty for firing on opposite-label evidence.

### Output fields

* `score`: final ranking score after token-evidence and selectivity penalties.
* `delta` / `effect`: raw and standardized positive-minus-negative activation differences.
* `pos_rate` / `neg_rate`: how often the feature fires in each label.
* `purity`: fraction of top contribution hits that come from the intended label.
* `hits`: compact text evidence for the ranked feature.
* `plot_examples`: contribution-ranked examples used in the activation plot.

Strict filters are applied first. Relaxed filters are only used to fill any missing slots. The final plots use contribution-ranked examples, so the highlighted tokens should directly explain why each feature was selected.

In [2]:
def rank_features(store, labels: list[int], tokenizer):
    """Return SAE features whose activations land on label-evidence tokens."""
    # Validate labels and activation shape before scoring.
    labels = [int(label) for label in labels]
    counts = {0: labels.count(0), 1: labels.count(1)}
    if set(labels) != {0, 1}:
        raise ValueError(f"SST-2 requires labels {{0, 1}}; got {counts}")

    acts = store.activations.float().cpu()
    if acts.dim() != 3 or acts.shape[0] != len(labels):
        raise ValueError(f"Expected [N, seq, n_features], got {acts.shape}")

    # Keep readable word-start tokens; GPT-2 continuation fragments make poor evidence.
    threshold_passes = (
        (0.5, 0.2, 0.55, 0.8),
        (0.3, 0.1, 0.75, 0.65),
        (0.05, 0.0, 1.0, 0.5),
        (float("-inf"), float("-inf"), 0.95, 0.4),
    )
    n_examples, seq, n_features = acts.shape
    labels_t = torch.tensor(labels, dtype=torch.long)
    tokens_cpu = store.tokens.cpu()
    mask = store.attention_mask.bool().cpu()
    if tokenizer.bos_token_id is not None:
        mask = mask & (tokens_cpu != tokenizer.bos_token_id)
    raw_tokens = [
        tokenizer.decode([int(token_id)])
        for token_id in tokens_cpu.reshape(-1).tolist()
    ]
    clean_tokens = [token.strip().lower() for token in raw_tokens]
    positions = torch.arange(seq).repeat(n_examples).tolist()
    content = torch.tensor(
        [
            (raw.startswith(" ") or pos == 0)
            and len(token) > 2
            and token not in STOPWORDS
            and any(char.isalpha() for char in token)
            for raw, token, pos in zip(raw_tokens, clean_tokens, positions, strict=True)
        ]
    ).reshape(mask.shape)
    mask = mask & content
    if not bool(mask.any()):
        raise ValueError("no readable content tokens remain after filtering")

    # Build smoothed label-token weights directly from this SST-2 slice.
    flat_mask = mask.reshape(-1)
    labels_by_token = labels_t[:, None].expand(n_examples, seq).reshape(-1)
    token_counts: dict[str, list[int]] = {}
    for token, label, ok in zip(
        clean_tokens, labels_by_token.tolist(), flat_mask.tolist(), strict=True
    ):
        if ok:
            token_counts.setdefault(token, [0, 0])[int(label)] += 1
    vocab = max(1, len(token_counts))
    totals = [sum(pair[label] for pair in token_counts.values()) for label in (0, 1)]
    label_weight = torch.zeros(n_examples * seq)
    for i, token in enumerate(clean_tokens):
        if not bool(flat_mask[i]):
            continue
        neg_count, pos_count = token_counts[token]
        log_odds = torch.log(torch.tensor((pos_count + 1) / (totals[1] + vocab)))
        log_odds -= torch.log(torch.tensor((neg_count + 1) / (totals[0] + vocab)))
        support = min(max(pos_count, neg_count), 4) / 4
        if abs(float(log_odds)) >= 0.5:
            label_weight[i] = float(log_odds) * support

    # Score features by label-token contribution plus strict sentence-level selectivity.
    token_counts_by_row = mask.sum(dim=1).clamp_min(1)
    pooled = (acts * mask.to(acts.dtype).unsqueeze(-1)).sum(dim=1)
    pooled = pooled / token_counts_by_row.to(acts.dtype).unsqueeze(-1)
    pos, neg = labels_t == 1, labels_t == 0
    pos_mean, neg_mean = pooled[pos].mean(0), pooled[neg].mean(0)
    delta = pos_mean - neg_mean
    effect = delta / pooled.std(0, unbiased=False).clamp_min(1e-6)
    pos_nz = (pooled[pos] > 0).sum(0)
    neg_nz = (pooled[neg] > 0).sum(0)
    pos_rate = pos_nz / pos.sum()
    neg_rate = neg_nz / neg.sum()
    flat_acts = acts.reshape(-1, n_features)

    rows = []
    per_sign = max(3, TOP_FEATURES // 2)
    for assoc, assoc_label, weight in (
        ("positive", 1, label_weight.clamp_min(0)),
        ("negative", 0, (-label_weight).clamp_min(0)),
    ):
        anchor_mask = flat_mask & (weight > 0)
        side_mask = anchor_mask & (labels_by_token == assoc_label)
        if not bool(side_mask.any()):
            continue
        class_gap = (pos_rate - neg_rate) if assoc_label else (neg_rate - pos_rate)
        other_rate = neg_rate if assoc_label else pos_rate
        signed_effect = effect if assoc_label else -effect
        anchor_score = (flat_acts[side_mask] * weight[side_mask, None]).sum(0)
        anchor_score = anchor_score / weight[side_mask].sum().clamp_min(1e-6)
        score_gap = class_gap.clamp_min(0.05)
        score_effect = signed_effect.clamp_min(0.05)
        base_score = anchor_score * score_gap * score_effect / (1 + other_rate)
        assoc_rows = []
        accepted_features = set()
        for min_effect, min_gap, max_other_rate, min_purity in threshold_passes:
            keep = (
                (signed_effect >= min_effect)
                & (class_gap >= min_gap)
                & (other_rate <= max_other_rate)
            )
            score = base_score.masked_fill(~keep, -torch.inf)

            # Keep strict passes first; later passes only fill missing slots.
            for feature_id in torch.argsort(score, descending=True)[: per_sign * 80]:
                feature_id = int(feature_id)
                if feature_id in accepted_features:
                    continue
                if (
                    not torch.isfinite(score[feature_id])
                    or float(score[feature_id]) <= 0
                ):
                    continue
                contrib = flat_acts[:, feature_id] * weight
                top_all = (
                    contrib.masked_fill(~anchor_mask, -torch.inf)
                    .topk(min(K_CONTEXTS, int(anchor_mask.sum())))
                    .indices
                )
                purity = float(
                    (labels_by_token[top_all] == assoc_label).float().mean().item()
                )
                vals, idxs = contrib.masked_fill(~side_mask, -torch.inf).topk(12)
                hits, plot_examples, seen_hits, seen_tokens = [], [], set(), set()
                for val, flat_i in zip(vals.tolist(), idxs.tolist(), strict=True):
                    if val <= 0:
                        continue
                    n, j = divmod(int(flat_i), seq)
                    token = tokenizer.decode([int(tokens_cpu[n, j])]).strip()
                    text = store.texts[n].strip()
                    key = (token.lower(), text)
                    if key in seen_hits:
                        continue
                    seen_hits.add(key)
                    seen_tokens.add(token.lower())
                    hits.append(f"{token!r} ({val:.2f}): {text}")

                    # Plot contribution evidence, not raw top activations.
                    valid = store.attention_mask[n].bool().cpu()
                    if tokenizer.bos_token_id is not None:
                        valid = valid & (tokens_cpu[n] != tokenizer.bos_token_id)
                    idx_map = [idx for idx, ok in enumerate(valid.tolist()) if ok]
                    ex_tokens = [
                        tokenizer.decode([int(tokens_cpu[n, idx])]) for idx in idx_map
                    ]
                    ex_values = [
                        float(acts[n, idx, feature_id] * float(weight[n * seq + idx]))
                        for idx in idx_map
                    ]
                    local_max = idx_map.index(j)
                    plot_examples.append(
                        {
                            "tokens": ex_tokens,
                            "activations": ex_values,
                            "max_activation": float(val),
                            "max_token": tokenizer.decode([int(tokens_cpu[n, j])]),
                            "max_activation_token_index": local_max,
                        }
                    )
                    if len(hits) == 3:
                        break
                if purity < min_purity or len(hits) < 2 or len(seen_tokens) < 2:
                    continue
                assoc_rows.append(
                    {
                        "feature_id": feature_id,
                        "assoc": assoc,
                        "score": float(score[feature_id]),
                        "delta": float(delta[feature_id]),
                        "effect": float(effect[feature_id]),
                        "pos_nz": int(pos_nz[feature_id]),
                        "neg_nz": int(neg_nz[feature_id]),
                        "pos_rate": float(pos_rate[feature_id]),
                        "neg_rate": float(neg_rate[feature_id]),
                        "purity": purity,
                        "hits": hits,
                        "plot_examples": plot_examples,
                    }
                )
                accepted_features.add(feature_id)
                if len(assoc_rows) >= per_sign:
                    break
            if len(assoc_rows) >= per_sign:
                break
        rows.extend(assoc_rows)

    rows = sorted(rows, key=lambda row: (row["assoc"], -row["score"]))
    for assoc in ("positive", "negative"):
        rank = 1
        for row in rows:
            if row["assoc"] == assoc:
                row["rank"] = rank
                rank += 1
    rows = sorted(rows, key=lambda row: (row["assoc"] != "positive", row["rank"]))
    return rows, {"0": int(counts[0]), "1": int(counts[1])}

In [3]:
# Load SST-2 and encode SAE activations through the existing Murano pipeline.
import warnings

from murano.model import MuranoModel
from murano.pipeline import Pipeline
from murano.steps import Load, SAEEncode

warnings.filterwarnings(
    "ignore",
    message=r"\s*This SAE has non-empty model_from_pretrained_kwargs\.",
    category=UserWarning,
    module=r"sae_lens\.saes\.sae",
)

dataset = LabeledDataset.from_hub(
    DATASET,
    text_column="sentence",
    label_column="label",
    split=SPLIT,
    n=N_EXAMPLES,
    label_names=["negative", "positive"],
)
model = MuranoModel(MODEL_ID)
sae_encode = SAEEncode(model, release=SAE_RELEASE, sae_id=SAE_ID)
results = Pipeline([Load(dataset), sae_encode]).run()

In [4]:
# Rank semantic sentiment features and save the same lightweight artifacts as the script.
from murano.steps import SAETopActivations, Save

rows, label_counts = rank_features(
    results["sae_record"], list(dataset.labels), model.tokenizer
)
results = Pipeline(
    [
        SAETopActivations(
            model, k=K_CONTEXTS, feat_ids=[row["feature_id"] for row in rows]
        ),
        Save(output_dir=OUTPUT_DIR, model_id=MODEL_ID),
    ]
).run(results)
contrast_path = Path(results["output_dir"]) / "sentiment_feature_contrast.json"
contrast_path.write_text(
    json.dumps(
        {
            "label_counts": label_counts,
            "score": "activation * label_token_weight * strict_class_selectivity",
            "features": rows,
        },
        indent=2,
    )
    + "\n"
)
feature_summary = [
    {
        "assoc": row["assoc"],
        "rank": row["rank"],
        "feature_id": row["feature_id"],
        "score": row["score"],
        "effect": row["effect"],
        "purity": row["purity"],
        "hits": row["hits"],
    }
    for row in rows
]
feature_summary

[{'assoc': 'positive',
  'rank': 1,
  'feature_id': 2259,
  'score': 0.017903897911310196,
  'effect': 0.24502712488174438,
  'purity': 1.0,
  'hits': ["'cinema' (18.60): this cinema verite speculation on the assassination of john f. kennedy may have been inspired by blair witch , but it takes its techniques into such fresh territory that the film never feels derivative",
   "'young' (13.30): as a young woman of great charm , generosity and diplomacy",
   "'stuff' (13.09): cool stuff packed into espn 's ultimate x."]},
 {'assoc': 'positive',
  'rank': 2,
  'feature_id': 16587,
  'score': 0.012702537700533867,
  'effect': 0.13122591376304626,
  'purity': 0.8999999761581421,
  'hits': ["'manages' (25.84): khouri manages , with terrific flair , to keep the extremes of screwball farce and blood-curdling family intensity on one continuum .",
   "'terrific' (25.37): is terrific as rachel",
   "'rock' (24.78): the rock 's fighting skills are more in line with steven seagal"]},
 {'assoc': 'pos

In [5]:
# Choose one highest-score positive feature and one highest-score negative feature.
positive_targets = sorted(
    (row for row in rows if row["assoc"] == "positive"),
    key=lambda row: row["score"],
    reverse=True,
)
negative_targets = sorted(
    (row for row in rows if row["assoc"] == "negative"),
    key=lambda row: row["score"],
    reverse=True,
)
if not positive_targets or not negative_targets:
    raise RuntimeError(
        f"Need positive and negative targets; got {len(positive_targets)} positive and {len(negative_targets)} negative."
    )
target_rows = positive_targets[:1] + negative_targets[:1]
[(row["assoc"], row["rank"], row["feature_id"], row["score"]) for row in target_rows]

[('positive', 1, 2259, 0.017903897911310196),
 ('negative', 1, 8293, 0.02445448562502861)]

In [6]:
# Extract raw decoder and unembedding matrices; plotting APIs own the projections/prep.
decoder = sae_encode.sae_model._sae.W_dec.detach().float().cpu()
unembedding = model._lm.lm_head.weight.detach().float().cpu()
token_labels = [
    model.tokenizer.decode([token_id]) for token_id in range(model.tokenizer.vocab_size)
]

In [7]:
# Build interactive figures for the targets; optionally export static PNGs.
SAVE_STATIC_PNG = False
figures = {}
png_files = []

for row in target_rows:
    current_feature_id = row["feature_id"]
    prefix = f"{row['assoc']}_rank{row['rank']}_feature{current_feature_id}"
    logit_fig = plot_sae_feature_logit_effects(
        current_feature_id,
        decoder=decoder,
        unembedding=unembedding,
        token_labels=token_labels,
        token_ids=list(range(model.tokenizer.vocab_size)),
        title=f"SST-2 SAE {row['assoc']} feature {current_feature_id} raw GPT-2 logit effects",
    )
    if "plot_examples" not in row:
        raise RuntimeError(
            "Rerun the ranking cell so rows include contribution plot examples."
        )
    activation_fig = plot_sae_token_activations(
        examples=row["plot_examples"],
        title=f"SST-2 SAE {row['assoc']} feature {current_feature_id} label-evidence token contributions",
    )
    figures[f"{prefix}_logit_effects"] = logit_fig
    figures[f"{prefix}_token_contributions"] = activation_fig
    row["logit_effects_figure"] = f"{prefix}_logit_effects"
    row["token_activations_figure"] = f"{prefix}_token_contributions"

    if SAVE_STATIC_PNG:
        logit_path = Path(results["output_dir"]) / f"{prefix}_logit_effects.png"
        activation_path = (
            Path(results["output_dir"]) / f"{prefix}_token_contributions.png"
        )
        logit_fig.write_image(logit_path)
        activation_fig.write_image(activation_path)
        png_files.extend([str(logit_path), str(activation_path)])

for row in target_rows:
    display(figures[row["logit_effects_figure"]])
    display(figures[row["token_activations_figure"]])
png_files or list(figures)

['positive_rank1_feature2259_logit_effects',
 'positive_rank1_feature2259_token_contributions',
 'negative_rank1_feature8293_logit_effects',
 'negative_rank1_feature8293_token_contributions']